In [16]:
from scraper import fetch_website_links, fetch_website_contents
from dotenv import load_dotenv
load_dotenv(override=True)
from IPython.display import Markdown, display, update_display
import json
from openai import OpenAI




In [17]:

MODEL = "llama3.1:8b"
client = OpenAI(
    base_url="http://localhost:11434/v1",  # ✅ correct ollama port + /v1
    api_key="ollama"                        # ✅ can be any non-empty string
)

In [18]:
link_system_prompt = """ 
you are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relvant to include in the brouchre about the company,
such as links to another page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:
{
    "Links": [
        {"type": "about page", "url":"https://full.url/goes/here/about"},
        {"type": "careers page", "url":"https://full.url/goes/here/careers"},
    ]
}
"""


In [19]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} - 
Please decide which of these are relevant links for a brocher about the company,
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy Policy, email links.

Links (some might be relative links):
"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt 


In [20]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type":"json_object"}
    )
    result =  response.choices[0].message.content
    try:
        links = json.loads(result)
        print(f"Found {len(links['Links'])} relevant links")
    except json.JSONDecodeError as e:
        print(f"Failed to parse JSON: {e}\nRaw result: {result}")
        raise
    
    return links

#print(select_relevant_links("https://edwarddonner.com/"))


In [21]:
#select_relevant_links("https://huggingface.co")

In [22]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing page:\n\n{contents}\n## Relevant links:\n"
    for link in relevant_links['Links']:
        result += f"\n\n### Link: {link["type"]}\n"
        result += fetch_website_contents(link["url"])
    return result
#print(fetch_page_and_all_relevant_links("https://edwarddonner.com/"))

In [23]:
brochure_system_prompt = """ 
You are an assistant that analyzes the contents of several revelant pages from a company website
and creates a short brochure about the company for prospective customers,investors and recruits to read.
Respond in a markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [24]:
def get_brochure_user_prompt(websitename, url):
    user_prompt = f"""
You are looking at a company called: {websitename}
Here are the contents of its landing page and other relevant pages:
use this information to create a brochure about the company in markdown without code blocks. \n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Limit to 5k characters for now
    return user_prompt 


In [ ]:
#get_brochure_user_prompt("edwarddonnar", "https://edwarddonner.com/" )

Selecting relevant links for https://edwarddonner.com/ by calling llama3.1:8b
Found 8 relevant links


'\nYou are looking at a company called: edwarddonnar\nHere are the contents of its landing page and other relevant pages:\nuse this information to create a brochure about the company in markdown without code blocks. \n\n\n## Landing page:\n\nHome - Edward Donner\n\nHome\nAI Curriculum\nProficient AI Engineer\nConnect Four\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of\nNebula.io\n. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. I’m previously the founder and CEO of AI startup untapt,\nacquired in 2021\n.\nI will happ

In [28]:
def create_brochure(websitename,url):
    response = client.chat.completions.create(
        model=MODEL,
        messages= [
            {"role":"system", "content":brochure_system_prompt},
            {"role":"user", "content":get_brochure_user_prompt(websitename,url)}
        ]
    )
    result = response.choices[0].message.content

    display(Markdown(result))

In [31]:
create_brochure("Apple" , "https://apple.com/")

Selecting relevant links for https://apple.com/ by calling llama3.1:8b
Found 4 relevant links


**Welcome to Apple, where Technology Meets Innovation**

at Apple, we believe in changing the world through technology that is accessible to everyone. Founded by Steve Jobs and Steve Wozniak, Apple has been revolutionizing the way people live, work, and play for over four decades.

**Our Mission:**

To bring the best user experience to our customers through innovative hardware, software, and services.

**Our Products:**

We design, manufacture, and market a wide range of products that integrate seamlessly with one another. Some of our most popular products include:

* iPhone: The world's most popular smartphone
* Mac: Laptops and desktop computers for work and play
* iPad: Tablets for entertainment and productivity
* Apple Watch: Wearable devices for fitness and health monitoring
* AirPods: Wireless earbuds with unparalleled sound quality

**Innovation Hub**

Apple is known for its commitment to innovation. We invest billions of dollars every year in research and development, focused on creating new technologies that improve people's lives.

**Community and Customer Relationships**

At Apple, we believe in building strong relationships with our customers and community partners. We offer a range of services, including:

* **Apple Support**: Expert help and resources for our products
* **Trade-In Programs**: Easy ways to upgrade or recycle your old devices
* **Apple Card**: A credit card that rewards you with cash back on every purchase

**Careers at Apple**

Are you passionate about technology and innovation? Join the Apple team as a part-time seasonal worker, contributing to the creation of amazing products that change people's lives.

**Why Work at Apple?**

We offer:

* Competitive salaries and benefits
* Opportunities for growth and development
* A dynamic work environment with talented colleagues from around the world

**Locations: Worldwide Operations**

Apple has a global presence with offices, stores, and data centers in over 20 countries worldwide. Join us on our mission to change the world through technology.

**Contact Us:**

We're here to help! Visit our website or contact us at [www.apple.com](http://www.apple.com) for more information about our products and services.

Come and experience the Apple difference for yourself.